# 03 · Downstream baseline

Fija la arquitectura CNN 1D con datos exclusivamente reales, para las dos tareas. A partir de este notebook la arquitectura queda congelada.

**Responsable:** Oscar

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/downstream/baseline_regimen.keras`
- `models/downstream/baseline_volatilidad.keras`
- `results/historiales/baseline_*.csv`
- `results/metricas/baseline.csv`
- `results/figures/aprendizaje_baseline_*.png`
- `results/figures/confusion_baseline.png`

**Tiempo estimado:** ~15 min en CPU (tres entrenamientos de hasta 60 épocas con parada temprana).

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import downstream, evaluacion, regimenes

n_regimenes = config.n_regimenes()
forma_entrada = train.X.shape[1:]

print("forma de entrada:", forma_entrada, "· clases:", n_regimenes)
print("train:", len(train), "· val:", len(val), "· test:", len(test))

## Lo que exige el enunciado

> *Usando los datos reales se buscará una arquitectura válida para resolver el
> problema. Usando la misma arquitectura se entrenarán distintas versiones del
> modelo, una versión por cada dataset generado.*

La comparación entre generadores solo es honesta si lo único que cambia entre
versiones son los datos. Por eso la arquitectura se elige aquí, con datos reales, y
después no se toca: cualquier ajuste posterior contaminaría el experimento, porque
no se sabría si la mejora viene del generador o del retoque.

In [ ]:
print(downstream.ARQUITECTURA)

modelo_regimen = downstream.construir("regimen", forma_entrada, n_clases=n_regimenes)
modelo_regimen.summary()

## Por qué esta troncal

Una CNN 1D sobre el eje temporal, en la línea de los cuadernos del máster. Cada
bloque convolucional reduce la longitud a la mitad y amplía los canales, de forma
que las primeras capas ven patrones locales (un salto de volatilidad de dos días) y
las últimas la forma global de la ventana.

Está dimensionada para CPU: con ~4.000 ventanas de (60, 20) una época cuesta del
orden de un segundo, y eso es lo que hace viable el barrido de varios cientos de
entrenamientos del notebook 12. Una arquitectura más grande daría métricas algo
mejores aquí y convertiría el barrido en inviable, que es el experimento que
realmente responde a la pregunta del taller.

El dropout es la única regularización explícita, y no es cosmético: los datasets
con mucho sintético son grandes pero poco diversos, y sin él el modelo memoriza las
muestras generadas.

## Tarea 1 · clasificación de régimen

La validación es siempre real, en todas las versiones del experimento. Solo el
conjunto de entrenamiento cambia entre recetas.

In [ ]:
historial_reg = downstream.entrenar(
    modelo_regimen,
    train.X, train.y_reg,
    val.X, val.y_reg,
    epocas=60, tam_lote=64, paciencia=12, verboso=0,
)
downstream.guardar_historial(historial_reg, "baseline_regimen")

metricas_reg = evaluacion.evaluar(modelo_regimen, test.X, test.y_reg, "regimen")
pd.Series(metricas_reg).round(4)

## Convergencia

Con parada temprana y restauración de los mejores pesos, la curva de validación
tiene que aplanarse antes del final. Si sigue bajando en la última época, el
presupuesto de épocas se ha quedado corto.

In [ ]:
historial = pd.read_csv(src.DIR_HISTORIALES / "baseline_regimen.csv")

fig, eje = plt.subplots()
viz.curva_convergencia(historial, "Aprendizaje · baseline de régimen (datos reales)", eje=eje)
viz.guardar(fig, "aprendizaje_baseline_regimen")

print("Épocas efectivas:", len(historial))

## Dónde falla el baseline

El accuracy engaña con estas clases: un modelo que nunca prediga crisis acierta el
90 % y es inútil. La fila de crisis de la matriz de confusión es la que importa, y
es la magnitud sobre la que los datos sintéticos deberían actuar.

In [ ]:
predicho = modelo_regimen.predict(test.X, verbose=0).argmax(axis=1)
matriz = evaluacion.matriz_confusion(test.y_reg, predicho, n_clases=n_regimenes)

fig, eje = plt.subplots(figsize=(6, 5))
viz.confusion(matriz, "Baseline de régimen · test real", eje=eje)
viz.guardar(fig, "confusion_baseline")

matriz

## Comparación obligada: reponderar la pérdida

Los pesos por clase son la alternativa clásica y gratuita al desbalance. Si
reponderar iguala al mejor generador, los datos sintéticos no aportan nada que no
se consiguiera sin ellos, y eso hay que decirlo en el informe. Este número es la
vara de medir del notebook 13.

In [ ]:
print("Pesos por clase:", downstream.pesos_por_clase(train.y_reg, n_regimenes))

modelo_pesos = downstream.construir("regimen", forma_entrada, n_clases=n_regimenes)
historial_pesos = downstream.entrenar(
    modelo_pesos, train.X, train.y_reg, val.X, val.y_reg,
    epocas=60, tam_lote=64, paciencia=12, verboso=0, usar_pesos=True,
)
downstream.guardar_historial(historial_pesos, "baseline_regimen_pesos")

metricas_pesos = evaluacion.evaluar(modelo_pesos, test.X, test.y_reg, "regimen")
pd.DataFrame({"sin pesos": metricas_reg, "con pesos": metricas_pesos}).round(4)

## Tarea 2 · regresión de volatilidad

Misma troncal, solo cambian la capa de salida y la pérdida. Que la troncal sea
idéntica permite atribuir cualquier diferencia entre tareas a la naturaleza del
problema y no a la arquitectura.

Se reporta QLIKE además de MAE porque es la pérdida estándar en previsión de
volatilidad: penaliza más infraestimar el riesgo que sobreestimarlo.

In [ ]:
modelo_vol = downstream.construir("volatilidad", forma_entrada, n_clases=n_regimenes)
historial_vol = downstream.entrenar(
    modelo_vol, train.X, train.y_vol, val.X, val.y_vol,
    epocas=60, tam_lote=64, paciencia=12, verboso=0,
)
downstream.guardar_historial(historial_vol, "baseline_volatilidad")

metricas_vol = evaluacion.evaluar(modelo_vol, test.X, test.y_vol, "volatilidad")
pd.Series(metricas_vol).round(4)

In [ ]:
historial = pd.read_csv(src.DIR_HISTORIALES / "baseline_volatilidad.csv")

fig, eje = plt.subplots()
viz.curva_convergencia(historial, "Aprendizaje · baseline de volatilidad (datos reales)", eje=eje)
eje.set_yscale("log")
viz.guardar(fig, "aprendizaje_baseline_volatilidad")

print("Épocas efectivas:", len(historial))

## Arquitectura congelada

A partir de aquí `downstream.ARQUITECTURA` no se toca. Cualquier cambio obliga a
reejecutar el notebook 12 entero —varios cientos de entrenamientos— y a rehacer
todas las figuras del informe.

Los dos modelos se guardan como referencia visual y de depuración; el barrido del
notebook 12 los reconstruye desde cero para cada receta, nunca los reutiliza.

In [ ]:
src.DIR_MODELOS_DOWN.mkdir(parents=True, exist_ok=True)

modelo_regimen.save(src.DIR_MODELOS_DOWN / "baseline_regimen.keras")
modelo_vol.save(src.DIR_MODELOS_DOWN / "baseline_volatilidad.keras")

filas = [
    {"variante": "regimen", "tarea": "regimen", **metricas_reg},
    {"variante": "regimen_pesos", "tarea": "regimen", **metricas_pesos},
    {"variante": "volatilidad", "tarea": "volatilidad", **metricas_vol},
]
evaluacion.acumular(filas, "baseline")

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_DOWN / "baseline_regimen.keras",
    src.DIR_MODELOS_DOWN / "baseline_volatilidad.keras",
    src.DIR_HISTORIALES / "baseline_regimen.csv",
    src.DIR_HISTORIALES / "baseline_regimen_pesos.csv",
    src.DIR_HISTORIALES / "baseline_volatilidad.csv",
    src.DIR_METRICAS / "baseline.csv",
    src.DIR_FIGURAS / "aprendizaje_baseline_regimen.png",
    src.DIR_FIGURAS / "aprendizaje_baseline_volatilidad.png",
    src.DIR_FIGURAS / "confusion_baseline.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
